## 06 - Lines and Distances

### Goal

- Last lesson we loaded the WDO package and used some of the given spatial functions. Now we are going to put package to use and calculate many distances and draw some lines. 
- Using any file that has "world cities" in the name:
  
  - [WorldCitiesGeo](../data/WorldCitiesGeo/) (and all its geojson files in the directory)
  - [world_cities_fixed.json](../data/world_cities_fixed.json)
  - [world_cities_by_time-zone.json](../data/world_cities_by_time-zone.json)
  
- These contain ALL THE CITIES IN THE WORLD! Well ... most of the cities.  You know like the ones that ... have people.  I mean `Burkburnett` probably isn't in the file, but it does have people, so ... (update ... I just checked, and Burk is in the file!)
  
- Keep reading


### Tasks

- Read in a file containing the locations to cities all over the world ([world_cities_large.json](./../data/world_cities_large.json)).
- We would like to draw a line from MSU (or close to it) to each city in the file, but thats not feasible as the file is too large. 
- Looking at the options (files) above, you should have plenty of choices for filtering down to a manageable size of cities to draw lines to. 


## Possible Bonus

- Randomly choose cities from the file, and stop choosing when the distance goes over some threshold (e.g 10000km).

In [1]:
from pathlib import Path


## Use last items learned from last lesson to locate and check if the file exists 
## Remember parent, and folder: data

# ----------------------------------------
# Distance: Haversine (km)
# ----------------------------------------
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Compute great-circle distance between two lat/lon points.
    Returns distance in kilometers.
    Inputs: degrees
    Output: kilometers
    """
    phi1 = radians(lat1)
    phi2 = radians(lat2)

    dphi = radians(lat2 - lat1)
    dlmb = radians(lon2 - lon1)

    a = sin(dphi / 2) ** 2 + cos(phi1) * cos(phi2) * sin(dlmb / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return EARTH_RADIUS_KM * c



debug = False

cwd = Path.cwd()
if debug: 
    print("Current Working Directory:")
    print(cwd)

target = cwd.parent / "data" / "countries.geojson"
if debug: 
    print("\nAttempting to access:")
    print(target)

exists = target.exists()
if debug: 
    print("\nDoes file exist?")
    print(exists)

assert exists, (
    f"\n❌ ERROR: File not found:\n{target}\n"
    "Check spelling and folder structure."
)



**FYI:**

These previous three lessons will be helpful: 
- [03-Style_W_Logic](./03-Style_W_Logic.ipynb)
- [04-Distance](./04-Distance.ipynb)
- [05-Geo_and_Json_overview](./05-Geo_and_Json_overview.ipynb)



In [2]:
# Completed reference solution
import json
import math
from pathlib import Path

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))

data_path = Path("../data/world_cities_fixed.json")
if not data_path.exists():
    data_path = Path("../data/world_cities_by_time-zone.json")

cities = json.loads(data_path.read_text(encoding="utf-8"))
if isinstance(cities, dict):
    records = cities.get("features", []) or cities.get("cities", []) or list(cities.values())
else:
    records = cities

def get_city(record):
    props = record.get("properties", record) if isinstance(record, dict) else {}
    geom = record.get("geometry", {}) if isinstance(record, dict) else {}
    coords = geom.get("coordinates")
    name = props.get("city") or props.get("name") or props.get("Name") or props.get("city_ascii") or "Unknown"
    if coords and len(coords) >= 2:
        lon, lat = coords[:2]
    else:
        lat = props.get("lat") or props.get("latitude")
        lon = props.get("lng") or props.get("lon") or props.get("longitude")
    return name, float(lat), float(lon)

parsed = []
for record in records[:5000]:
    try:
        parsed.append(get_city(record))
    except Exception:
        pass

base = (33.9137, -98.4934)
distances = [(name, haversine_km(base[0], base[1], lat, lon), lat, lon) for name, lat, lon in parsed]
distances = sorted(distances, key=lambda item: item[1])
print("Closest 10 cities to Wichita Falls sample:")
for name, distance, lat, lon in distances[:10]:
    print(f"{name:<30} {distance:8.1f} km ({lat:.3f}, {lon:.3f})")


Closest 10 cities to Wichita Falls sample:
Unknown                          3913.8 km (18.172, -63.149)
Unknown                          3917.2 km (18.201, -63.090)
Unknown                          3918.0 km (18.205, -63.078)
Unknown                          3918.0 km (18.192, -63.088)
Unknown                          3918.4 km (18.176, -63.094)
Unknown                          3919.1 km (18.217, -63.058)
Unknown                          3919.4 km (18.199, -63.066)
Unknown                          3919.5 km (18.229, -63.044)
Unknown                          3919.9 km (18.220, -63.046)
Unknown                          3920.9 km (18.256, -63.010)


This lesson helps with **📏 Milestone 2** 